# Plot the output of cellpose_GFP_RFP

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)
library(rstatix)
library(emmeans)
library(tibble)
library(tidyr)


options(tibble.width = Inf)

## Plot parameters

In [ ]:
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/Figures/GFP_RFP/output"
filter_dapi = TRUE

analysis_summary_files = c(
"/ceph.groups/mshahbazi.grp/rsakata/EXP48/plots/summarised_results.csv",
"/ceph.groups/mshahbazi.grp/rsakata/EXP49/output/batch1/plots/summarised_results.csv",
"/ceph.groups/mshahbazi.grp/rsakata/EXP49/output/batch2/plots/summarised_results.csv",
"/ceph.groups/mshahbazi.grp/rsakata/EXP49/output/batch3/plots/summarised_results.csv",
"/ceph.groups/mshahbazi.grp/rsakata/EXP50/plots/summarised_results.csv"
)

In [ ]:
# define const for visualization
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.5/2.141959

# Set geom defaults globally
update_geom_defaults("line",      list(linewidth = LINE.W))
update_geom_defaults("errorbar",  list(linewidth = LINE.W))
#update_geom_defaults("point",     list(size = LINE.W, stroke = LINE.W))

settheme <- theme_minimal() + 
  theme(
    text = element_text(family = "sans"), 
    panel.background = element_blank(),
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"), 
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(size = FONT.SIZE),
    axis.text = element_text(colour = "black", size = FONT.SIZE),
    strip.text = element_text(size = FONT.SIZE), 
    strip.text.y.left = element_text(angle = 0, hjust = 1, size = FONT.SIZE),
    legend.position = "right",
    legend.title = element_text(size = FONT.SIZE), 
    legend.text = element_text(size = FONT.SIZE),
    legend.key.size = unit(0.3, "cm"),
    axis.text.x = element_text(colour = "black", angle = 0, size = LABEL.FONT.SIZE),
    title = element_text(size = FONT.SIZE) 
  )

In [ ]:
col_condition = c("G_R"= "#5E5E5E","Grev_R"="#86AB30","Rrev_G"="#EB5951", "Grev_Rrev"="#F0A329")
col_GFP = "#86AB30"
col_RFP = "#EB5951"

## 1. Extract summary files

In [ ]:
merged_df <- analysis_summary_files %>%
  map_dfr(read_csv)

In [ ]:
head(merged_df)

In [ ]:
unique(merged_df$sample_name)

In [ ]:
tbl <- merged_df %>%
  group_by( sample_name, timepoint, state) %>%
  summarise(n_images = n_distinct(image), n_EXP = n_distinct(EXP), .groups = "drop") 
tbl

In [ ]:
df_sample = merged_df

In [ ]:
order_cond <- c("G_R","Grev_Rrev",  "Grev_R", "Rrev_G")

df_sample <- df_sample %>%
  mutate(condition = factor(condition, levels = order_cond))

# Plot 

In [ ]:
order_sample <- c(
  "D4_GR",  "D4_GrevR", "D4_RrevG", "D4_GrevRrev",
  "D6_GR_D", "D6_GR_F", "D6_GrevR_D", "D6_GrevR_F", "D6_RrevG_D", "D6_RrevG_F","D6_GrevRrev_D", "D6_GrevRrev_F"
  )
df_sample <- df_sample %>%
  mutate(sample_name = factor(sample_name, levels = order_sample))


In [ ]:
order_condition <- c(
  "G_R",  "Grev_R", "Rrev_G", "Grev_Rrev"
  )
df_sample <- df_sample %>%
  mutate(condition = factor(condition, levels = order_condition))

In [ ]:
head(df_sample)

In [ ]:
df_sample <- df_sample %>%
  mutate(
    condition2 = if_else(
      condition %in% c("Grev_R", "Rrev_G"),
      "mosaic",
      condition
    )
  )

### A) total cell numbers

In [ ]:
title = "counts_per_structure_labelled"
w <- 3
h <- 1.8
options(repr.plot.width=w, repr.plot.height=h)
  
p = ggplot(df_sample, aes(x = condition, y =total_labelled)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
    geom_jitter(
    color = "#5E5E5E",
    position = position_jitter(width = 0.05, height = 0),
    size = 0.5,
    alpha = 0.8
  ) +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "number of cells",
      x = ""
    )+
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
      facet_wrap(~ state, scales = "free_y") +
      scale_color_manual(values=col_condition)+ 
      settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) 

ggsave(file.path(out_dir, sprintf("A_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
df_test <- df_sample %>%
  filter(!is.na(state)) %>%
  mutate(
    state = factor(state),
    condition = factor(condition)
  )

anova_res <- df_test %>%
  group_by(state) %>%
  anova_test(total_labelled ~ condition)

anova_res

In [ ]:
dunnett_res <- df_test %>%
  group_by(state) %>%
  group_modify(~ {
    fit <- aov(total_labelled ~ condition, data = .x)

    emm <- emmeans(fit, ~ condition)

    out <- contrast(
      emm,
      method = "trt.vs.ctrl",
      ref = "G_R"   # change this to your control condition
    )

    as.data.frame(out)
  })

dunnett_res

In [ ]:
title = "counts_per_structure_labelled"
w <- 3
h <- 1.8
options(repr.plot.width=w, repr.plot.height=h)

order_condition <- c(
  "G_R",  "mosaic", "Grev_Rrev"
  )
df_sample <- df_sample %>%
  mutate(condition2 = factor(condition2, levels = order_condition))

  
p = ggplot(df_sample, aes(x = condition2, y =total_labelled)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
    geom_jitter(
    color = "#5E5E5E",
    position = position_jitter(width = 0.05, height = 0),
    size = 0.5,
    alpha = 0.8
  ) +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "number of cells",
      x = ""
    )+
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
      facet_wrap(~ state, scales = "free_y") +
      scale_color_manual(values=col_condition)+ 
      settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) 

ggsave(file.path(out_dir, sprintf("A_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
wilcox_res <- df_test %>%
  group_by(state) %>%
  wilcox_test(
    total_labelled ~ condition,
    ref.group = "G_R",
    p.adjust.method = "holm"
  ) %>%
  mutate(
    stars = case_when(
      p.adj < 0.0001 ~ "****",
      p.adj < 0.001  ~ "***",
      p.adj < 0.01   ~ "**",
      p.adj < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

In [ ]:
dunn_res <- df_test %>%
  group_by(state) %>%
  dunn_test(
    total_labelled ~ condition2,
    p.adjust.method = "holm"
  )
dunn_res 

In [ ]:
kw_res <- df_test %>%
  group_by(state) %>%
  kruskal_test(total_labelled ~ condition2) %>%
  mutate(
    stars = case_when(
      p < 0.0001 ~ "****",
      p < 0.001  ~ "***",
      p < 0.01   ~ "**",
      p < 0.05   ~ "*",
      TRUE       ~ "ns"
    )
  )
kw_res

In [ ]:
title = "counts_per_structure_labelled"
w <- 3
h <- 1.8
options(repr.plot.width=w, repr.plot.height=h)

order_condition <- c(
  "G_R",  "mosaic", "Grev_Rrev"
  )
df_sample <- df_sample %>%
  mutate(condition2 = factor(condition2, levels = order_condition))

  
p = ggplot(df_sample, aes(x = condition2, y =total_labelled)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
    geom_jitter(
    color = "#5E5E5E",
    position = position_jitter(width = 0.05, height = 0),
    size = 0.5,
    alpha = 0.8
  ) +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "number of cells",
      x = ""
    )+
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
      facet_wrap(~ state, scales = "free_y") +
      scale_color_manual(values=col_condition)+ 
      settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) 

ggsave(file.path(out_dir, sprintf("A_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
wilcox_res <- df_test %>%
  group_by(state) %>%
  wilcox_test(
    total_labelled ~ condition2,
    ref.group = "G_R",
    p.adjust.method = "holm"
  ) %>%
  mutate(
    stars = case_when(
      p.adj < 0.0001 ~ "****",
      p.adj < 0.001  ~ "***",
      p.adj < 0.01   ~ "**",
      p.adj < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

### B) Percentage GFP+

In [ ]:
title = "per_GFPpos"
w <- 3
h <- 2
options(repr.plot.width=w, repr.plot.height=h)
  
p = ggplot(df_sample, aes(x = condition, y =propGFP , group = condition)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      fill = "grey", alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = condition),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_GFP 
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%EGFP+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
      
    ) +
      scale_y_continuous(limits = c(0, 1.1), expand = c(0, 0))+
      facet_wrap( ~ state) 
      #scale_color_manual(values=col_sample_name)

ggsave(file.path(out_dir, sprintf("B_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "per_GFPsub"
w <- 1.6
h <- 1.6
options(repr.plot.width=w, repr.plot.height=h)

df_sample_sub = 
df_sample %>%
  subset(condition %in% c("G_R", "Grev_R"))

  
p = ggplot(df_sample_sub, aes(x = condition, y =propGFP , group = condition)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      fill = "grey", alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = condition),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_GFP 
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%EGFP+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
      
    ) +
      scale_y_continuous(limits = c(0, 1.1), expand = c(0, 0))+
      facet_wrap( ~ state) 
      #scale_color_manual(values=col_sample_name)

ggsave(file.path(out_dir, sprintf("B_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
library(dplyr)
library(tidyr)
library(purrr)

check_test <- function(data, group_var = "condition", value_var = "propGFP",
                        conditions = c("G_R", "Grev_R"), alpha = 0.05) {

  d <- data %>%
    filter(.data[[group_var]] %in% conditions) %>%
    droplevels()

  d %>%
    group_by(state) %>%
    group_modify(~ {
      g <- split(.x[[value_var]], .x[[group_var]])
      g <- g[conditions]                       # keep the two groups in order

      # need at least 3 non-NA points per group for Shapiro
      n_ok <- all(sapply(g, function(x) sum(!is.na(x)) >= 3))

      if (!n_ok) {
        return(tibble(
          n1 = sum(!is.na(g[[1]])), n2 = sum(!is.na(g[[2]])),
          shapiro_p1 = NA_real_, shapiro_p2 = NA_real_,
          levene_p = NA_real_, normal = NA,
          recommended = "too few points (use Wilcoxon / be cautious)"
        ))
      }

      # normality per group
      sp1 <- shapiro.test(g[[1]])$p.value
      sp2 <- shapiro.test(g[[2]])$p.value
      normal <- (sp1 > alpha) & (sp2 > alpha)

      # equal-variance check (F-test; swap for car::leveneTest if preferred)
      var_p <- tryCatch(var.test(g[[1]], g[[2]])$p.value, error = function(e) NA_real_)

      rec <- if (normal) {
        if (!is.na(var_p) && var_p > alpha) "Student t-test (var.equal = TRUE)"
        else "Welch t-test"
      } else {
        "Wilcoxon rank-sum test"
      }

      tibble(
        n1 = sum(!is.na(g[[1]])), n2 = sum(!is.na(g[[2]])),
        shapiro_p1 = sp1, shapiro_p2 = sp2,
        levene_p = var_p, normal = normal,
        recommended = rec
      )
    }) %>%
    ungroup()
}

# usage
check_test(df_sample)

In [ ]:
wilcox_res <- df_sample_sub %>%
  group_by(state) %>%
  wilcox_test(
    propGFP ~ condition,
    #ref.group = "G_R",
    p.adjust.method = "BH"
  ) %>%
  mutate(
    p_use = if ("p.adj" %in% names(.)) p.adj else p,   # fall back to raw p
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

In [ ]:
title = "per_GFPpos_D6"
w <- 3
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

df_sample_sub = df_sample %>% subset(timepoint == "D6")
  
p = ggplot(df_sample_sub, aes(x = condition, y =propGFP , group= state)) +  # dots for each file
    stat_summary( aes(fill = state), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = state),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_GFP
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%EGFP+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 0, hjust = 0.5)
    ) +
      scale_y_continuous(limits = c(0, 1.1), expand = c(0, 0))+
  scale_fill_manual(
    values = c(Developed = "grey50", Failed = "grey80"),
    name = "state"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

### C) GFP+ counts

In [ ]:
title = "count_GFPpos"
w <- 3
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

  
p = ggplot(df_sample, aes(x = condition, y =GFPpos )) +  # dots for each file
    stat_summary( 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6, fill= "grey") +   # error bars
    geom_jitter(
      aes(fill = condition),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_GFP
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "number of EGFP+ cells",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) +
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
      facet_wrap( ~ state) 


ggsave(file.path(out_dir, sprintf("C_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
check_test <- function(data, group_var = "condition", value_var = "GFPpos",
                       conditions = c("G_R", "Grev_R", "Grev_Rrev"), alpha = 0.05) {

  d <- data %>%
    filter(.data[[group_var]] %in% conditions) %>%
    mutate(across(all_of(group_var), ~ factor(.x, levels = conditions))) %>%
    droplevels()

  d %>%
    group_by(state) %>%
    group_modify(~ {
      g <- split(.x[[value_var]], .x[[group_var]])[conditions]
      k <- length(g)
      ns <- sapply(g, function(x) sum(!is.na(x)))

      if (!all(ns >= 3)) {
        return(tibble(
          k = k, n = paste(ns, collapse = "/"),
          shapiro_p_min = NA_real_, levene_p = NA_real_, normal = NA,
          recommended = "too few points (use non-parametric / be cautious)"
        ))
      }

      # normality: test every group, take the worst
      sps <- sapply(g, function(x) shapiro.test(x)$p.value)
      normal <- all(sps > alpha)

      # homogeneity of variance across all k groups (Levene, robust to non-normality)
      lev_p <- tryCatch(
        car::leveneTest(.x[[value_var]] ~ .x[[group_var]])[1, "Pr(>F)"],
        error = function(e) NA_real_
      )
      equal_var <- !is.na(lev_p) && lev_p > alpha

      rec <- if (k == 2) {
        if (normal) { if (equal_var) "Student t-test" else "Welch t-test" }
        else "Wilcoxon rank-sum test"
      } else {
        if (normal) { if (equal_var) "One-way ANOVA + Tukey HSD"
                      else "Welch's ANOVA + Games-Howell" }
        else "Kruskal-Wallis + Dunn's test"
      }

      tibble(
        k = k, n = paste(ns, collapse = "/"),
        shapiro_p_min = min(sps), levene_p = lev_p, normal = normal,
        recommended = rec
      )
    }) %>%
    ungroup()
}

check_test(df_sample)

In [ ]:
head(df_sample)

In [ ]:
kw_res <- df_sample %>%
  filter(condition %in% c("G_R", "Grev_R", "Grev_Rrev"))%>%
  group_by(state) %>%
  kruskal_test(GFPpos ~ condition) %>%
  mutate(
    stars = case_when(
      p < 0.0001 ~ "****",
      p < 0.001  ~ "***",
      p < 0.01   ~ "**",
      p < 0.05   ~ "*",
      TRUE       ~ "ns"
    )
  )
kw_res

In [ ]:
dunn_res <- df_sample  %>%
  filter(condition %in% c("G_R", "Grev_R", "Grev_Rrev"))%>%
  group_by(state) %>%
  dunn_test(
    GFPpos ~ condition,
    p.adjust.method = "BH"
  )

dunn_res 

### D) RFP+ 

In [ ]:
title = "per_RFPpos"
w <- 3
h <- 2
options(repr.plot.width=w, repr.plot.height=h)
  
p = ggplot(df_sample, aes(x = condition, y =propRFP , group = condition)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      fill = "grey", alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = condition),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_RFP 
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%mcherry+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
      
    ) +
      scale_y_continuous(limits = c(0, 1.1), expand = c(0, 0))+
      facet_wrap( ~ state) 
      #scale_color_manual(values=col_sample_name)

ggsave(file.path(out_dir, sprintf("D_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "per_RFPpos_labelled_D6"
w <- 5
h <- 2.2
options(repr.plot.width=w, repr.plot.height=h)

df_sample_sub = df_sample %>% subset(timepoint == "D6")
  
p = ggplot(df_sample_sub, aes(x = condition, y =propRFP , group= state)) +  # dots for each file
    stat_summary( aes(fill = state), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = state),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_RFP
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%mcherry+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 0, hjust = 0.5)
    ) +
      scale_y_continuous(limits = c(0, 1.1), expand = c(0, 0))+
  scale_fill_manual(
    values = c(Developed = "grey50", Failed = "grey80"),
    name = "state"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "per_RFPsub"
w <- 1.6
h <- 1.6
options(repr.plot.width=w, repr.plot.height=h)

df_sample_sub = 
df_sample %>%
  subset(condition %in% c("G_R", "Rrev_G"))

  
p = ggplot(df_sample_sub, aes(x = condition, y =propRFP , group = condition)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      fill = "grey", alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = condition),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_RFP
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%mcherry+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
      
    ) +
      scale_y_continuous(limits = c(0, 1.1), expand = c(0, 0))+
      facet_wrap( ~ state) 
      #scale_color_manual(values=col_sample_name)

ggsave(file.path(out_dir, sprintf("B_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
wilcox_res <- df_sample_sub %>%
  group_by(condition) %>%
  wilcox_test(
    propRFP ~ state,
    #ref.group = "G_R",
    p.adjust.method = "BH"
  ) %>%
  mutate(
    p_use = if ("p.adj" %in% names(.)) p.adj else p,   # fall back to raw p
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

### E) RFP + counts

In [ ]:
title = "count_RFPpos"
w <- 3
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

  
p = ggplot(df_sample, aes(x = condition, y =RFPpos )) +  # dots for each file
    stat_summary( 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6, fill= "grey") +   # error bars
    geom_jitter(
      aes(fill = condition),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_RFP
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "number of mcherry+ cells",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) +
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
      facet_wrap( ~ state) 


ggsave(file.path(out_dir, sprintf("E_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
kw_res <- df_sample %>%
  filter(condition %in% c("G_R", "Rrev_G", "Grev_Rrev"))%>%
  group_by(state) %>%
  kruskal_test(RFPpos ~ condition) %>%
  mutate(
    stars = case_when(
      p < 0.0001 ~ "****",
      p < 0.001  ~ "***",
      p < 0.01   ~ "**",
      p < 0.05   ~ "*",
      TRUE       ~ "ns"
    )
  )
kw_res

In [ ]:
dunn_res <- df_sample  %>%
  filter(condition %in% c("G_R", "Rrev_G", "Grev_Rrev"))%>%
  group_by(state) %>%
  dunn_test(
    RFPpos ~ condition,
    p.adjust.method = "BH"
  )

dunn_res 

In [ ]:
title = "count_RFPpos_labelled_D6"
w <- 2
h <- 1.5
options(repr.plot.width=w, repr.plot.height=h)

df_sample_D6 = df_sample %>% subset(state == "Developed")%>% subset(condition %in%  c("G_R", "Grev_Rrev", "Rrev_G"))
  
p = ggplot(df_sample_D6, aes(x = condition, y =RFPpos )) +  # dots for each file
    geom_jitter(
      aes(fill = condition),
      position = position_jitterdodge(jitter.width = 0.4, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_RFP
    ) + stat_summary( 
      fun = mean, 
      geom = "point",
      shape = 95,
      size = 4,    
      position = position_dodge(width = 0.75),
      alpha = 1, width = 1, fill= "grey") +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.1, 
      color = "black")+
    labs(
      title = title,
      y = "number of mcherry+ cells",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "count_GFPpos_labelled_D6"
w <- 2
h <- 1.5
options(repr.plot.width=w, repr.plot.height=h)

df_sample_D6 = df_sample %>% subset(state == "Developed")%>% subset(condition %in%  c("G_R", "Grev_Rrev", "Grev_R"))
  
p = ggplot(df_sample_D6, aes(x = condition, y =GFPpos )) +  # dots for each file
    geom_jitter(
      aes(fill = condition),
      position = position_jitterdodge(jitter.width = 0.4, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_GFP
    ) + stat_summary( 
      fun = mean, 
      geom = "point",
      shape = 95,
      size = 4,    
      position = position_dodge(width = 0.75),
      alpha = 1, width = 1, fill= "grey") +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.1, 
      color = "black")+
    labs(
      title = title,
      y = "number of GFP+ cells",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
df_sample_D4 %>%
  group_by(state) %>%
  t_test(
    RFPpos ~ condition,
    p.adjust.method = "none"
  ) 

In [ ]:
title = "count_RFPpos_labelled_D6"
w <- 5
h <- 2.2
options(repr.plot.width=w, repr.plot.height=h)

df_sample_sub = df_sample %>% subset(timepoint == "D6")
  
p = ggplot(df_sample_sub, aes(x = condition, y =RFPpos , group= state)) +  # dots for each file
    stat_summary( aes(fill = state), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = state),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = col_RFP
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "number of mcherry+ cells",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 0, hjust = 0.5)
    ) +
      scale_y_continuous(limits = c(0, 470), expand = c(0, 0))+
  scale_fill_manual(
    values = c(Developed = "grey50", Failed = "grey80"),
    name = "state"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
write_csv(df_sample, file.path(out_dir, "summarised_results.csv"))